# Cuneiform Stoicheia: restoring and transliterating cuneiform, sign by sign

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericu9500/cuneiform-stoicheia/blob/main/notebooks/cuneiform_stoicheia_demo.ipynb)

This notebook shows what the model does, on tablets it has never seen. The model reads a tablet as a sequence
of *signs* (entries of the Oracc Sign List), with their readings, word boundaries and determinatives, and was
trained on about 120,000 texts from Oracc and CDLI to recover signs that are hidden from it.

Every tablet used below is held out: neither it, nor another exemplar of the same composition, nor a tablet
with the same museum number, nor a near-identical text was in the training data.

Contents: (1) a tablet as the model sees it; (2) one lost sign; (3) a break of known width; (4) a break of
unknown width; (5) real breaks, checked against a duplicate; (6) transliterating bare signs; (7) how far the
model's confidence can be trusted; (8) your own line.

The notebook runs on a CPU; with a GPU (Runtime → Change runtime type) it is faster.

In [ ]:
# The model repository on the Hugging Face Hub (or a local folder with the same files).
REPO = "Ericu950/cuneiform-stoicheia"

In [ ]:
import importlib.util, os, subprocess, sys
missing = [p for p in ("torch", "numpy", "pandas", "safetensors", "huggingface_hub") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

REPO = os.environ.get("CUNEIFORM_STOICHEIA_REPO", REPO)
token = None
try:                                    # while the repository is private: a token named HF_TOKEN in Colab's secrets
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if os.path.isdir(REPO):
    path = REPO
else:
    from huggingface_hub import snapshot_download
    path = snapshot_download(REPO, token=token)
sys.path.insert(0, path)

import numpy as np, pandas as pd, torch
from IPython.display import HTML, display
from cuneiform_stoicheia import Restorer

m = Restorer.from_pretrained(path)
V = m.vocab
tablets = {t["id"]: t for t in Restorer.examples(path, "tablets")}
lacunae = Restorer.examples(path, "lacunae")
print(f"model on {m.device}; {len(tablets)} held-out tablets, {len(lacunae)} real lacunae")

In [ ]:
# Display helpers.
CSS = '''<style>@import url("https://fonts.googleapis.com/css2?family=Noto+Sans+Cuneiform&display=swap");
.cun {font-family: "Noto Sans Cuneiform", "Segoe UI Historic", sans-serif; font-size: 150%; white-space: nowrap}
table.cs td, table.cs th {text-align: left !important; vertical-align: top; padding: 3px 10px}
.hit {background: #1baf7a33} .gap {background: #eb683433; font-weight: 600}</style>'''

def table(rows, columns, cun=()):
    # rows of cells -> HTML table; columns named in `cun` are set in a cuneiform font
    head = "".join(f"<th>{c}</th>" for c in columns)
    body = ""
    for r in rows:
        cls = r.pop() if isinstance(r[-1], dict) else {}
        tds = "".join(f'<td class="{"cun" if c in cun else ""}">{x}</td>' for c, x in zip(columns, r))
        body += f'<tr class="{cls.get("class", "")}">{tds}</tr>'
    display(HTML(CSS + f'<table class="cs"><tr>{head}</tr>{body}</table>'))

def load(tid):
    t = tablets[tid]
    return t, m.from_arrays(t["lines"], t["lang"])

def clean_runs(T, n):
    # starts of stretches of n signs that are identified and undamaged in the edition
    ok = (T.sign >= 6) & (T.status == 0)
    return [i for i in range(len(T) - n + 1) if ok[i:i + n].all()]

def tr(T, sign, read, join, det):
    return V.translit(sign, read, join, det, line_sep=" / ")

## 1. A tablet as the model sees it

An Ur III administrative tablet from the held-out set. The edition gives the transliteration; the model works
on the signs behind it, shown here in Unicode cuneiform (□ marks a sign without a code point). Damage marks
follow the edition: `#` a damaged sign, `[ ]` a sign restored by the editor, `x` an illegible sign.

In [ ]:
t, T = load("P499279")
print(t["id"], "|", t["period"], "|", t["genre"], "|", t["lang"], "|", len(T), "signs")
table([[k + 1, T.translit(a, b), T.cuneiform(a, b), " ".join(V.sign_name[s] for s in T.sign[a:b])]
       for k, (a, b) in enumerate(T.lines())], ["line", "transliteration", "cuneiform", "sign names"], cun=("cuneiform",))

All tablets shipped with the model:

In [ ]:
pd.DataFrame([{"id": t["id"], "language": t["lang"], "period": t["period"], "genre": t["genre"],
               "lines": len(t["lines"]), "signs": sum(len(l[0]) for l in t["lines"])} for t in tablets.values()]).fillna("")

## 2. One lost sign

We hide one sign of the tablet, together with its reading, and ask for the ten most probable signs. The
model sees the rest of the tablet on both sides. Each candidate comes with a probability and with the reading
the model would give the sign in this place. The sign that the tablet really has is highlighted.

In [ ]:
def one_sign(T, pos, k=10):
    h = T.hide(pos, 1)
    truth = int(h.truth[0][0][0])
    a, b = next((a, b) for a, b in T.lines() if a <= pos < b)
    print("line:   ", T.translit(a, pos), "⟦", T.translit(pos, pos + 1), "⟧", T.translit(pos + 1, b), "\n")
    cands = m.top_signs(h, k=k)
    table([[i + 1, f"{c.prob:.3f}", c.names, c.cuneiform, c.translit,
            {"class": "hit" if int(c.sign[0]) == truth else ""}] for i, c in enumerate(cands)],
          ["rank", "probability", "sign", "cuneiform", "reading here"], cun=("cuneiform",))

rng = np.random.default_rng(1)
positions = rng.choice(clean_runs(T, 1), 3, replace=False)
for p in sorted(positions):
    one_sign(T, int(p))

## 3. A break of known width

Now several signs in a row are lost, as in `[x x x]`. The model fills the most certain position first, feeds
its choice back and continues; a beam search keeps several fillings alive, so the result is a ranked list of
whole restorations. The probability is that of the whole filling. Signs, readings, word boundaries and
determinatives are all predicted; only the signs are compared with the tablet.

Three tablets of different kinds, each with one break of three and one of six signs.

In [ ]:
def known_gap(tid, n, seed=0, beam=20, topk=5):
    t, T = load(tid)
    start = int(np.random.default_rng(seed).choice(clean_runs(T, n)))
    h = T.hide(start, n)
    truth = h.truth[0]
    lo = max(0, start - 8); hi = min(len(T), start + n + 8)
    print(f"{t['id']} ({t['period']}, {t['genre']}, {t['lang']})")
    print("  context: ", T.translit(lo, start, " / "), "⟦", n, "signs lost ⟧", T.translit(start + n, hi, " / "))
    cands = m.restore(h, beam=beam, topk=topk)
    rows = [["tablet", "", tr(T, *truth), V.cuneiform(truth[0]), "", {"class": "gap"}]]
    for i, c in enumerate(cands):
        right = int((c.sign == truth[0]).sum())
        rows.append([i + 1, f"{c.prob:.3f}", c.translit, c.cuneiform, f"{right}/{n}",
                     {"class": "hit" if right == n else ""}])
    table(rows, ["rank", "probability", "restoration", "cuneiform", "signs right"], cun=("cuneiform",))

for tid in ("P499279", "P336516", "P388305"):
    for n in (3, 6):
        known_gap(tid, n, seed=n)

The pattern seen here is the general one: formulaic stretches (dates, standard phrases, lists of commodities)
are restored well; names, numbers and wording without a parallel are where the model fails, and it fails with
plausible-looking text. Section 7 comes back to how to tell the two apart.

## 4. A break of unknown width

Often the editor knows only roughly how much is lost: `[...]`. The model is then given a number of *slots*,
the most signs the break can hold (in training: the true width plus a margin of up to a third). It tries every
width up to that number, fills each, and keeps the width whose filling is most probable. The table shows the
score of every width and the best fillings at the chosen width.

In [ ]:
def unknown_gap(tid, n, slots, seed=0):
    t, T = load(tid)
    start = int(np.random.default_rng(seed).choice(clean_runs(T, n)))
    h = T.hide(start, n, unknown_slots=slots)
    truth = h.truth[0]
    lo = max(0, start - 8); hi = min(len(T), start + n + 8)
    print(f"{t['id']} ({t['period']}, {t['genre']})")
    print("  context: ", T.translit(lo, start, " / "), "⟦ ... ⟧", T.translit(start + n, hi, " / "))
    cands = m.restore(h, beam=10, topk=3)
    ws = cands.info["width_scores"]
    print("  true width:", n, "| slots given:", slots, "| chosen width:", cands.info["width"])
    print("  score per width (log-probability, higher is better):", {k: round(v, 1) for k, v in ws.items()})
    rows = [["tablet", "", tr(T, *truth), V.cuneiform(truth[0]), {"class": "gap"}]]
    rows += [[i + 1, f"{c.prob:.3f}", c.translit, c.cuneiform,
              {"class": "hit" if np.array_equal(c.sign, truth[0]) else ""}] for i, c in enumerate(cands)]
    table(rows, ["rank", "probability", "restoration", "cuneiform"], cun=("cuneiform",))

unknown_gap("P499279", 2, slots=4, seed=3)
unknown_gap("P336516", 4, slots=7, seed=4)
unknown_gap("P388305", 3, slots=6, seed=5)

Width is the harder half of this task. In the evaluation, most of what the model loses when the width is
not given comes from choosing the wrong number of signs. If the width of a break can be estimated from the
tablet, give it.

## 5. Real breaks, checked against a duplicate

The breaks above were made by hiding preserved text. Here nothing is simulated. Each case is a manuscript
(A) with a real break, for which another manuscript (B) of the same composition preserves the passage. The
model sees A with its break and is asked for as many signs as B has there; neither manuscript was in the
training data. Where A's editor restored the break, that restoration is shown too.

First one case in detail, then all forty.

In [ ]:
def lacuna_text(x):
    T = m.from_arrays(x["context"], x["lang"])
    T.gaps = [(x["start"], x["end"] - x["start"], "known")]
    return T

def show_lacuna(x, topk=5):
    T = lacuna_text(x)
    s, e = x["start"], x["end"]
    ans = [np.array(r) for r in x["answer"]]
    print(f"A = {x['a_id']}  B = {x['b_id']}  composition {', '.join(x['composites']) or '-'}  ({x['period']}, {x['genre']})")
    print("  A as edited: ", T.translit(max(0, s - 10), s, " / "), "⟦", T.translit(s, e, " / "), "⟧",
          T.translit(e, min(len(T), e + 10), " / "))
    cands = m.restore(T, 0, slots=x["L"], unknown=False, beam=20, topk=topk)
    rows = [["B preserves", "", tr(T, *ans), V.cuneiform(ans[0]), {"class": "gap"}]]
    if x["editor"] is not None:
        rows.append(["A's editor", "", " ".join(V.sign_name[i] for i in x["editor"]), V.cuneiform(x["editor"]), {}])
    rows += [[i + 1, f"{c.prob:.3f}", c.translit, c.cuneiform,
              {"class": "hit" if np.array_equal(c.sign, ans[0]) else ""}] for i, c in enumerate(cands)]
    table(rows, ["", "probability", "text", "cuneiform"], cun=("cuneiform",))

show_lacuna(next(x for x in lacunae if 3 <= x["L"] <= 5 and x["editor"] is not None))
show_lacuna(next(x for x in lacunae if 2 <= x["L"] <= 4 and x["kind"] != "restored"))

In [ ]:
rows, n1, n10 = [], 0, 0
for x in lacunae:
    T = lacuna_text(x)
    ans = np.array(x["answer"][0])
    cands = m.restore(T, 0, slots=x["L"], unknown=False, beam=10, topk=10)
    rank = next((i + 1 for i, c in enumerate(cands) if np.array_equal(c.sign, ans)), None)
    n1 += rank == 1; n10 += rank is not None
    rows.append([x["a_id"], x["lang"], x["L"], tr(T, *[np.array(r) for r in x["answer"]]), cands[0].translit,
                 "–" if rank is None else rank, {"class": "hit" if rank == 1 else ""}])
table(rows, ["manuscript", "language", "signs", "B preserves", "model, first candidate", "rank of B's text"])
print(f"exactly B's text as first candidate: {n1} of {len(lacunae)}; among the first ten: {n10} of {len(lacunae)}")

Two cautions on reading this table. B is another manuscript, not the lost surface of A: manuscripts vary, so
a restoration that differs from B is not always wrong for A. And these forty are an illustration; the
measured rates over the full sets are in the model card and the technical report.

## 6. Transliterating bare signs

The second task: only the signs are given, as a sign list or from a copy, and the model supplies the reading
of every sign, the word boundaries and the determinatives. Each sign's reading is chosen among the readings
attested for that sign. The model's line is set beside the edition's.

In [ ]:
def transliterate(tid, max_lines=12):
    t, T = load(tid)
    out = m.transliterate(T.sign, t["lang"])
    use = (T.sign >= 6) & (T.read > 1)
    agree = (out["read"][use] == T.read[use])
    rows = []
    for k, (a, b) in enumerate(T.lines()[:max_lines]):
        ed = V.translit(T.sign[a:b], T.read[a:b], T.join[a:b], T.det[a:b])
        mo = V.translit(T.sign[a:b], out["read"][a:b], out["join"][a:b], out["det"][a:b])
        rows.append([k + 1, T.cuneiform(a, b), mo, ed, {"class": "" if mo == ed else "gap"}])
    print(f"{t['id']} ({t['period']}, {t['genre']}, {t['lang']}): {agree.sum()} of {agree.size} readings as in the edition "
          f"({100 * agree.mean():.1f}%)")
    table(rows, ["line", "signs", "model", "edition"], cun=("signs",))

transliterate("P499279")
transliterate("P336516")
transliterate("P388305")

Highlighted lines differ from the edition somewhere: in a reading, a word boundary, a determinative or the
division into lines (which the model also predicts). Some differences are errors; others are conventions on
which editions differ, such as `pu₂` or `tul₂`, `ud` or `u₄`, `{m}` or `{diš}` for the same sign. The
percentage counts only identical readings, so it is a lower bound. The same call accepts sign names or
cuneiform characters:

In [ ]:
print(m.transliterate("𒀭 𒂗 𒆤", lang="sux")["translit"])
print(m.transliterate("LUGAL KAL GA", lang="sux")["translit"])

## 7. How far can the confidence be trusted?

For a single lost sign the model's probability for its first candidate is a usable guide. We hide 400 signs,
one at a time, across all the example tablets, and compare the stated probability with how often the first
candidate is right.

In [ ]:
rng = np.random.default_rng(7)
cases = []
for tid in tablets:
    t, T = load(tid)
    runs = clean_runs(T, 1)
    for p in rng.choice(runs, min(9, len(runs)), replace=False):
        cases.append(T.hide(int(p), 1))
cases = cases[:400]
res = m.top_signs(cases, k=10, readings=False)
conf = np.array([r[0].prob for r in res])
right = np.array([int(r[0].sign[0]) == int(h.truth[0][0][0]) for r, h in zip(res, cases)])
in10 = np.array([any(int(c.sign[0]) == int(h.truth[0][0][0]) for c in r) for r, h in zip(res, cases)])
print(f"{len(cases)} signs: first candidate right {100 * right.mean():.1f}%, among the first ten {100 * in10.mean():.1f}%")
bins = [(0, .3), (.3, .5), (.5, .7), (.7, .9), (.9, .99), (.99, 1.01)]
pd.DataFrame([{"stated probability": f"{a:.2f}–{min(b, 1):.2f}", "cases": int(((conf >= a) & (conf < b)).sum()),
               "first candidate right (%)": round(100 * right[(conf >= a) & (conf < b)].mean(), 1) if ((conf >= a) & (conf < b)).any() else None}
              for a, b in bins])

This suggests a simple rule for abstaining: accept the first candidate only above a threshold, and otherwise
leave the sign open (or look at the list of candidates).

In [ ]:
pd.DataFrame([{"accept if probability ≥": th, "signs answered (%)": round(100 * (conf >= th).mean(), 1),
               "of those, right (%)": round(100 * right[conf >= th].mean(), 1) if (conf >= th).any() else None}
              for th in (0.0, 0.5, 0.8, 0.9, 0.95, 0.99)])

This holds for one sign with the rest of the text preserved. Inside a longer break it does not: the signs are
filled one after another, each on the assumption that the earlier ones are right, so the probabilities of the
later signs are too high. Treat the probability of a long restoration as a ranking of the candidates, not as
a chance of being right.

## 8. Your own line

Type a transliteration in ATF, as CDLI writes it (`sz`, `s,`, `t,`, index digits) or as Oracc does (`š`, `ṣ`,
`ṭ`, subscripts). One line of the tablet per line; give as much of the tablet as you have, since the model
uses the context.

- `x` is one lost or illegible sign; a run of them is a break of known width: `[x x x]`
- `...` is a break of unknown width; `slots` below is the most signs it may hold
- `{d}`, `{ki}` are determinatives, `#` marks a damaged sign, signs inside `[ ]` count as restored
- `lang` is `"sux"` for Sumerian, `"akk"` for Akkadian

In [ ]:
text = '''
1(disz) sila4 ur-mes ensi2
1(disz) sila4 [x x] sukkal
mu-kux(DU) [...]
iti ezem-{d}nin-a-zu
mu en {d}nanna [x x x] ba-hun
'''
lang = "sux"
slots = 4

T = m.parse(text, lang)
print(T.translit(), "\n")
for i, (start, n, kind) in enumerate(T.gaps):
    cands = m.restore(T, i, slots=slots if kind == "unknown" else None, beam=20, topk=5)
    what = f"{n} sign(s)" if kind == "known" else f"unknown width, at most {slots} signs (chosen: {cands.info['width']})"
    print(f"gap {i + 1}: {what}")
    table([[k + 1, f"{c.prob:.3f}", c.translit, c.names, c.cuneiform] for k, c in enumerate(cands)],
          ["rank", "probability", "restoration", "signs", "cuneiform"], cun=("cuneiform",))
best, filled = m.restore_all(T, slots=slots)
print("with the first candidate in every gap (« »):\n")
print(filled)

## Notes

- The model proposes; the editor decides. A restoration is a suggestion to be weighed against the tablet,
  the parallels and the grammar.
- It sees up to 256 signs of context and nothing of the object: no photograph, no traces, no line position.
- Numerals and names are its weakest points, and new wording is much harder than formulaic text.
- Model, vocabulary and example tablets: CC BY-SA 4.0; code: MIT. The texts come from Oracc projects and from
  CDLI; when quoting a tablet, cite its edition.

Author: Eric Cullhed. Code and technical report: <https://github.com/ericu9500/cuneiform-stoicheia>.